In [1]:
#from torch import nn
import pandas as pd
#import numpy as np
from pathlib import Path
import pandera as pa
from nfl.lib import enums

from nfl.data_management.DataManager import DataManager
from sklearn.cluster import HDBSCAN

In [2]:
data_path = (Path.cwd() / "nfl/data").resolve()

In [3]:
data = DataManager.get_data(path_to_json = data_path)

reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2015.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2005.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2016.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2006.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_1999.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2023.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2008.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2014.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2017.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2009.csv
reading /home/peter/personal/my_machine_learning/nfl_analyti

In [23]:
EXCLUDED_PLAY_TYPES = {
    enums.PlayType.KICK,
    enums.PlayType.EXTRA_POINT,
    enums.PlayType.NO_PLAY,
    enums.PlayType.GAME_START,
}

data = data[
            data["play_type"].notna()   
            & ~data["play_type"].isin(EXCLUDED_PLAY_TYPES)]
pd.set_option('display.max_columns', None)

data["has_roof"] = data["roof"].isin(
    [enums.RoofType.DOME, enums.RoofType.CLOSED]
)

data["has_turf"] = data["surface"] != enums.SurfaceType.GRASS

gap_map = {"guard": 1, "tackle": 2, "end": 3}
data["run_gap"] = data["run_gap"].map(gap_map).fillna(-5).astype(int)

data.head(5)

,yardline_100,game_seconds_remaining,quarter_end,drive,sp,qtr,down,goal_to_go,ydstogo,ydsnet,play_type,yards_gained,shotgun,no_huddle,qb_dropback,qb_kneel,qb_spike,qb_scramble,pass_length,air_yards,yards_after_catch,run_gap,field_goal_result,extra_point_result,score_differential,score_differential_post,ep,epa,wp,home_wp,wpa,home_wp_post,air_wpa,yac_wpa,comp_air_wpa,comp_yac_wpa,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,fourth_down_converted,fourth_down_failed,incomplete_pass,touchback,interception,fumble_forced,fumble_not_forced,fumble_out_of_bounds,solo_tackle,safety,penalty,tackled_for_loss,fumble_lost,qb_hit,rush_attempt,pass_attempt,sack,touchdown,pass_touchdown,rush_touchdown,return_touchdown,extra_point_attempt,field_goal_attempt,fumble,complete_pass,assist_tackle,passing_yards,receiving_yards,rushing_yards,tackle_with_assist,fumble_recovery_1_yards,fumble_recovery_2_yards,return_yards,penalty_yards,replay_or_challenge,replay_or_challenge_result,penalty_type,season,cp,cpoe,series,series_success,series_result,order_sequence,play_clock,play_type_nfl,st_play_type,end_yard_line,drive_play_count,drive_first_downs,drive_ended_with_score,drive_quarter_start,drive_quarter_end,drive_yards_penalized,drive_start_transition,drive_end_transition,drive_game_clock_start,drive_game_clock_end,drive_start_yard_line,drive_end_yard_line,result,div_game,roof,surface,temp,wind,aborted_play,success,pass,rush,first_down,special,play,out_of_bounds,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe,day_of_season,has_roof,has_turf
2,80.0,3600.0,False,1.0,False,1,1.0,False,10,18.0,pass,3.0,False,False,True,False,False,False,short,3.0,0.0,-5,NaN,NaN,0.0,0.0,0.239785,-0.337139,0.422024,0.577976,-0.001425,0.579401,-0.001425,0.000000,-0.001425,0.000000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,0.0,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,3.0,3.0,NaN,False,NaN,NaN,0.0,NaN,False,NaN,NaN,2015,0.765811,23.418927,1,True,First down,51.0,12.0,PASS,NaN,23.0,6.0,1.0,False,1.0,1.0,0.0,KICKOFF,PUNT,15:00,11:33,20.0,38.0,6,False,outdoors,grass,88.0,13.0,False,False,True,False,False,False,True,False,4.699278,3.0,0.678964,0.225919,0.456481,54.351911,43,False,False
3,77.0,3573.0,False,1.0,False,1,2.0,False,7,18.0,run,2.0,False,False,False,False,False,False,NaN,NaN,NaN,-5,NaN,NaN,0.0,0.0,-0.097354,-0.262481,0.420599,0.579401,-0.017304,0.596705,NaN,NaN,0.000000,0.000000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,0.0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN,2.0,False,NaN,NaN,0.0,NaN,False,NaN,NaN,2015,NaN,NaN,1,True,First down,75.0,18.0,RUSH,NaN,25.0,6.0,1.0,False,1.0,1.0,0.0,KICKOFF,PUNT,15:00,11:33,20.0,38.0,6,False,outdoors,grass,88.0,13.0,False,False,False,True,False,False,True,False,NaN,NaN,NaN,NaN,0.545905,-54.590458,43,False,False
4,75.0,3532.0,False,1.0,False,1,3.0,False,5,18.0,pass,10.0,True,False,True,False,False,False,short,4.0,6.0,-5,NaN,NaN,0.0,0.0,-0.359835,1.661242,0.403295,0.596705,0.045358,0.551347,0.000000,0.045358,0.000000,0.045358,False,True,False,True,False,False,False,False,False,False,False,False,False,True,0.0,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,10.0,10.0,NaN,False,NaN,NaN,0.0,NaN,False,NaN,NaN,2015,0.510176,48.982388,1,True,First down,96.0,5.0,PASS,NaN,35.0,6.0,1.0,False,1.0,1.0,0.0,KICKOFF,PUNT,15:00,11:33,20.0,38.0,6,False,outdoors,grass,88.0,13.0,False,True,True,False,True,False,True,False,4.662650,2.0,0.712350,0.712350,0.968533,3.146732,43,False,False
5,65.0,3494.0,False,1.0,False,1,1.0,False,10,18.0,run,0.0,False,False,False,False,False,False,NaN,NaN,NaN,-5,NaN,NaN,0.0,0.0,1.301407,-0.518931,0.448653,0.551347,-0.018066,0.569413,NaN,NaN,0.000000,0.000000,False,False,False,False,False,False,False,False,False,False,False,False

In [ ]:
scenario_columns = [
    "yardline_100",
    "game_seconds_remaining",
    "has_turf",
    "temp",
    "wind",
    "has_roof",
    "ydstogo",
    "goal_to_go",
    "score_differential",
    "down",
    "div_game",
    "day_of_season",
    "series"
]

play_columns = [
    "play_type",
    "pass_location",
#    "pass_length",
#    "run_location",
    "run_gap",
    "shotgun",
    "no_huddle",
    "qb_kneel",
    "qb_spike",
    "qb_scramble",
    "air_yards"
]

result_columns = [
    "epa",
    "wpa",
    "success",
    "result",
    "series_success",
    "tackle_for_loss",
    "saftey",
    "yards_gained",
    "touchdown",
    "fumble",
    "complete_pass",
    "rushing_yards",
    "fumble_lost",
    "interception",
    "sack",
    "penalty_yards",
]

for col in data.columns:
    #if col in scenario_columns or col in play_columns:
    print(f"{col}: {data[col].dtype}")

data["play_type"].unique()

yardline_100: float64
game_seconds_remaining: float64
quarter_end: boolean
drive: float64
sp: boolean
qtr: int64
down: float64
goal_to_go: boolean
ydstogo: int64
ydsnet: float64
play_type: str
yards_gained: float64
shotgun: boolean
no_huddle: boolean
qb_dropback: boolean
qb_kneel: boolean
qb_spike: boolean
qb_scramble: boolean
pass_length: str
air_yards: float64
yards_after_catch: float64
run_gap: int64
field_goal_result: str
extra_point_result: str
score_differential: float64
score_differential_post: float64
ep: float64
epa: float64
wp: float64
home_wp: float64
wpa: float64
home_wp_post: float64
air_wpa: float64
yac_wpa: float64
comp_air_wpa: float64
comp_yac_wpa: float64
first_down_rush: boolean
first_down_pass: boolean
first_down_penalty: boolean
third_down_converted: boolean
third_down_failed: boolean
fourth_down_converted: boolean
fourth_down_failed: boolean
incomplete_pass: boolean
touchback: boolean
interception: boolean
fumble_forced: boolean
fumble_not_forced: boolean
fumble_o

<StringArray>
['pass', 'run', 'qb_kneel', 'qb_spike']
Length: 4, dtype: str

In [ ]:
incomplete_with_air_yards_df = data[
    (data["incomplete_pass"] == True) & (data["air_yards"] > 0)
]

In [31]:
has_incomplete_with_air_yards

np.True_

In [25]:
data.info()

<class 'pandas.DataFrame'>
Index: 887392 entries, 2 to 1245910
Columns: 124 entries, yardline_100 to has_turf
dtypes: bool(2), boolean(52), float64(48), int64(7), object(2), str(13)
memory usage: 570.4+ MB


In [ ]:
bool_cols = data.select_dtypes(include=['bool', 'boolean']).columns
for col in bool_cols:
    data[col] = data[col].astype('Int64')

numeric_df = data.select_dtypes(include=['number']).fillna(0)

# 2. Find top columns correlated with 'epa' (excluding epa itself)
# Change 'top_n' to include more or fewer features
top_n = 10
correlations = numeric_df.corr()['epa'].abs().drop('epa')
top_features = correlations.nlargest(top_n).index

In [ ]:
top_features

In [ ]:
bool_cols = data.select_dtypes(include=['bool', 'boolean']).columns
for col in bool_cols:
    data[col] = data[col].astype('Int64')
X = data.select_dtypes(include=['number']).fillna(0)
X = X[top_features]
labels = HDBSCAN(min_cluster_size=10).fit_predict(X)


In [ ]:
with pd.option_context('display.max_rows', None):
    print(data.iloc[0])

In [ ]:
data["result"].unique()

In [ ]:
data["series_result", "fixed_drive_result"].head()

In [ ]:
all_df["surface_type"].unique()

In [ ]:
cleaner = DataCleaner("schema.json")
try:
    clean_df = cleaner.clean(all_df)
except pa.errors.SchemaErrors as err:
    print(err.failure_cases)

In [ ]:
clean_df.head()

In [ ]:
all_df["weather"].unique()